In [1]:
from pyspark.sql import SparkSession, functions as F

spark = SparkSession.builder.master("local[*]").appName("champignons").getOrCreate()

df = spark.read.csv("data/mushrooms.csv", header=True)
# Les tirets dans les noms de colonnes posent problème en SQL : on les remplace
df = df.toDF(*[c.replace("-", "_") for c in df.columns])

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/10/27 11:37:42 WARN Utils: Your hostname, tuxedo, resolves to a loopback address: 127.0.1.1; using 192.168.20.29 instead (on interface wlp132s0f0)
26/10/27 11:37:42 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/10/27 11:37:42 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [2]:
df.printSchema()
df.show(5)
print(df.count())          # 8124 attendus

root
 |-- class: string (nullable = true)
 |-- cap_shape: string (nullable = true)
 |-- cap_surface: string (nullable = true)
 |-- cap_color: string (nullable = true)
 |-- bruises: string (nullable = true)
 |-- odor: string (nullable = true)
 |-- gill_attachment: string (nullable = true)
 |-- gill_spacing: string (nullable = true)
 |-- gill_size: string (nullable = true)
 |-- gill_color: string (nullable = true)
 |-- stalk_shape: string (nullable = true)
 |-- stalk_root: string (nullable = true)
 |-- stalk_surface_above_ring: string (nullable = true)
 |-- stalk_surface_below_ring: string (nullable = true)
 |-- stalk_color_above_ring: string (nullable = true)
 |-- stalk_color_below_ring: string (nullable = true)
 |-- veil_type: string (nullable = true)
 |-- veil_color: string (nullable = true)
 |-- ring_number: string (nullable = true)
 |-- ring_type: string (nullable = true)
 |-- spore_print_color: string (nullable = true)
 |-- population: string (nullable = true)
 |-- habitat: string 

In [3]:
df.groupBy("class").count().show()   # e = comestible, p = toxique

+-----+-----+
|class|count|
+-----+-----+
|    e| 4208|
|    p| 3916|
+-----+-----+



In [4]:
# Nombre de valeurs différentes par colonne : veil_type n'en a qu'une, elle ne sert à rien
df.agg(*[F.countDistinct(c).alias(c) for c in df.columns]).show(vertical=True)

-RECORD 0-----------------------
 class                    | 2   
 cap_shape                | 6   
 cap_surface              | 4   
 cap_color                | 10  
 bruises                  | 2   
 odor                     | 9   
 gill_attachment          | 2   
 gill_spacing             | 2   
 gill_size                | 2   
 gill_color               | 12  
 stalk_shape              | 2   
 stalk_root               | 5   
 stalk_surface_above_ring | 4   
 stalk_surface_below_ring | 4   
 stalk_color_above_ring   | 9   
 stalk_color_below_ring   | 9   
 veil_type                | 1   
 veil_color               | 4   
 ring_number              | 3   
 ring_type                | 5   
 spore_print_color        | 9   
 population               | 6   
 habitat                  | 7   



In [5]:
df.filter(F.col("stalk_root") == "?").count()   # les valeurs manquantes

2480

In [6]:
df.createOrReplaceTempView("champignons")
spark.sql("""
    SELECT odor, class, COUNT(*) AS n
    FROM champignons
    GROUP BY odor, class
    ORDER BY odor
""").show(50)

+----+-----+----+
|odor|class|   n|
+----+-----+----+
|   a|    e| 400|
|   c|    p| 192|
|   f|    p|2160|
|   l|    e| 400|
|   m|    p|  36|
|   n|    p| 120|
|   n|    e|3408|
|   p|    p| 256|
|   s|    p| 576|
|   y|    p| 576|
+----+-----+----+



In [7]:
df = df.drop("veil_type")
# Cible en nombre : toxique = 1, comestible = 0
df = df.withColumn("label", (F.col("class") == "p").cast("double"))

features = [c for c in df.columns if c not in ("class", "label")]

train, test = df.randomSplit([0.8, 0.2], seed=42)
train.cache(); test.cache()
print(train.count(), test.count())

6573 1551


In [9]:
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler

idx_cols = [c + "_idx" for c in features]
ohe_cols = [c + "_ohe" for c in features]

indexer = StringIndexer(inputCols=features, outputCols=idx_cols, handleInvalid="keep")
encoder = OneHotEncoder(inputCols=idx_cols, outputCols=ohe_cols, handleInvalid="keep")
assembler = VectorAssembler(inputCols=ohe_cols, outputCol="features")

In [10]:
from pyspark.ml.classification import (
    LogisticRegression, DecisionTreeClassifier, RandomForestClassifier, GBTClassifier,
)
from pyspark.ml.evaluation import MulticlassClassificationEvaluator, BinaryClassificationEvaluator

def evaluer(pred):
    m = MulticlassClassificationEvaluator(labelCol="label")
    return {
        "accuracy": m.evaluate(pred, {m.metricName: "accuracy"}),
        "f1": m.evaluate(pred, {m.metricName: "f1"}),
        "rappel_toxiques": m.evaluate(pred, {m.metricName: "recallByLabel", m.metricLabel: 1.0}),
        "auc": BinaryClassificationEvaluator(labelCol="label").evaluate(pred),
    }

modeles = {
    "regression_logistique": LogisticRegression(maxIter=50),
    "arbre": DecisionTreeClassifier(maxDepth=5, seed=42),
    "foret": RandomForestClassifier(numTrees=100, seed=42),
    "gbt": GBTClassifier(maxIter=50, seed=42),
}

resultats = {}
for nom, modele in modeles.items():
    pipeline_model = Pipeline(stages=[indexer, encoder, assembler, modele]).fit(train)
    resultats[nom] = evaluer(pipeline_model.transform(test))

for nom, r in resultats.items():
    print(nom, {k: round(v, 4) for k, v in r.items()})

26/10/27 11:44:51 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


regression_logistique {'accuracy': 1.0, 'f1': 1.0, 'rappel_toxiques': 1.0, 'auc': 1.0}
arbre {'accuracy': 0.9929, 'f1': 0.9929, 'rappel_toxiques': 0.9948, 'auc': 0.989}
foret {'accuracy': 0.9865, 'f1': 0.9865, 'rappel_toxiques': 0.9725, 'auc': 0.9998}
gbt {'accuracy': 1.0, 'f1': 1.0, 'rappel_toxiques': 1.0, 'auc': 1.0}


In [11]:
pred = Pipeline(stages=[indexer, encoder, assembler, modeles["arbre"]]).fit(train).transform(test)
pred.groupBy("label", "prediction").count().show()

+-----+----------+-----+
|label|prediction|count|
+-----+----------+-----+
|  1.0|       1.0|  759|
|  0.0|       1.0|    7|
|  1.0|       0.0|    4|
|  0.0|       0.0|  781|
+-----+----------+-----+



In [12]:
from pyspark.ml.tuning import ParamGridBuilder, CrossValidator

rf = RandomForestClassifier(seed=42)
grille = (ParamGridBuilder()
          .addGrid(rf.numTrees, [20, 50, 100])
          .addGrid(rf.maxDepth, [5, 10])
          .build())

cv = CrossValidator(
    estimator=Pipeline(stages=[indexer, encoder, assembler, rf]),
    estimatorParamMaps=grille,
    evaluator=MulticlassClassificationEvaluator(labelCol="label", metricName="f1"),
    numFolds=3,
    seed=42,
)
cv_model = cv.fit(train)
meilleur = cv_model.bestModel
print(evaluer(meilleur.transform(test)))

{'accuracy': 1.0, 'f1': 1.0, 'rappel_toxiques': 1.0, 'auc': 1.0}


In [13]:
attrs = meilleur.transform(test).schema["features"].metadata["ml_attr"]["attrs"]
noms = {a["idx"]: a["name"] for groupe in attrs.values() for a in groupe}
imp = meilleur.stages[-1].featureImportances
for score, nom in sorted(((imp[i], noms[i]) for i in noms), reverse=True)[:10]:
    print(round(score, 3), nom)

0.114 gill_size_ohe_n
0.067 gill_size_ohe_b
0.064 odor_ohe_f
0.064 stalk_surface_above_ring_ohe_k
0.059 ring_type_ohe_l
0.059 odor_ohe_n
0.047 stalk_surface_below_ring_ohe_k
0.042 gill_spacing_ohe_w
0.038 spore_print_color_ohe_h
0.027 bruises_ohe_f


In [14]:
meilleur.write().overwrite().save("models/champignons_rf")

from pyspark.ml import PipelineModel
recharge = PipelineModel.load("models/champignons_rf")
recharge.transform(test).select("class", "prediction", "probability").show(5)

+-----+----------+--------------------+
|class|prediction|         probability|
+-----+----------+--------------------+
|    e|       0.0|           [1.0,0.0]|
|    e|       0.0|[0.99921826141338...|
|    e|       0.0|           [1.0,0.0]|
|    e|       0.0|[0.99921826141338...|
|    e|       0.0|           [1.0,0.0]|
+-----+----------+--------------------+
only showing top 5 rows
